#### Data Analysis project

#### >> data collection
#### >> data cleaning
#### >> EDA (Exploratory Data Analysis)
#### >> insight and conclusion -->


In [1]:
# India ecommerce sales and customer analysis using Python

# India E-Commerce Sales & Customer Analytics Using Python

**An end-to-end exploratory data analysis of 15,000+ online orders placed across 18 Indian states**

---

| | |
|---|---|
| **Domain** | Retail / E-Commerce Analytics |
| **Tools** | Python, NumPy, Pandas, Matplotlib, Seaborn, Plotly |
| **Dataset** | `india_ecommerce_sales.csv` (synthetic, 15,120 rows x 29 columns) |
| **Period covered** | 01-Jan-2024 to 31-Dec-2025 (24 months) |
| **Currency** | Indian Rupee (INR) |
| **Deliverables** | This notebook, a PDF report, a PowerPoint deck and exported charts |

---

## 1. Business Problem

**BharatKart Retail Pvt. Ltd.** is a mid-sized Indian e-commerce marketplace selling across
eight categories in 25 cities. Over the last two years the company has grown its top line
aggressively through heavy festive discounting and rapid category expansion.

Management is now facing an uncomfortable situation:

> *"Revenue keeps growing, but profit is not growing at the same pace. We do not know which
> categories, which regions and which customers are actually making money for us."*

The leadership team cannot answer basic questions such as *which state is most profitable*,
*how much margin our discounts destroy*, or *why our customer ratings are falling in some
regions*. Decisions are being taken on intuition rather than evidence.

This project analyses two full years of transactional data to convert that raw order log
into a clear, quantified picture of where the business makes money, where it loses money,
and what management should do next.

## 2. Project Objectives

1. Load, inspect and **clean** a realistic (deliberately messy) transactional dataset.
2. Engineer time, customer and pricing features that make the data analysis-ready.
3. Compute the **core KPIs** management needs on a single page.
4. Run a structured **exploratory data analysis** (univariate, bivariate, multivariate).
5. Quantify **which categories, products, regions and customers drive revenue and profit**.
6. Measure the **impact of discounting** on profitability.
7. Measure the **impact of delivery performance** on customer satisfaction.
8. Map performance **geographically across India**.
9. Segment customers using an **RFM** model.
10. Convert every finding into **actionable business recommendations**.

## 3. Dataset Description

The file `data/india_ecommerce_sales.csv` is a **synthetic but statistically realistic**
order log. Every row is one order line.

### Data dictionary

| Column | Type | Description |
|---|---|---|
| `Order_ID` | object | Unique identifier of the order |
| `Order_Date` | date | Date the order was placed |
| `Customer_ID` | object | Unique identifier of the customer |
| `Customer_Name` | object | Customer full name |
| `Gender` | object | Male / Female |
| `Age` | float | Age of the customer in years |
| `Age_Group` | object | Age bucket (18-25, 26-35, 36-45, 46-60, 60+) |
| `City` | object | Delivery city |
| `State` | object | Delivery state |
| `Region` | object | North / South / East / West / Central / Northeast |
| `Latitude` | float | Latitude of the city (for mapping) |
| `Longitude` | float | Longitude of the city (for mapping) |
| `Product_ID` | object | Unique identifier of the product |
| `Product_Name` | object | Product name |
| `Category` | object | One of 8 merchandising categories |
| `Sub_Category` | object | Sub-category within the category |
| `Quantity` | int | Units ordered |
| `Unit_Price` | float | Listed price per unit before discount (INR) |
| `Discount_Percentage` | float | Discount applied on the order (%) |
| `Sales` | float | Net booked revenue after discount (INR) |
| `Cost` | float | Cost of goods sold (INR) |
| `Profit` | float | Sales minus Cost, adjusted for returns (INR) |
| `Profit_Margin` | float | Profit as a percentage of Sales |
| `Payment_Method` | object | UPI, Credit Card, COD, EMI, ... |
| `Order_Status` | object | Delivered / Shipped / Returned / Cancelled |
| `Delivery_Days` | float | Days between order and delivery |
| `Customer_Rating` | float | Post-delivery rating on a 1-5 scale |
| `Customer_Segment` | object | Premium / Regular / Budget / Corporate |
| `Device_Type` | object | Channel used to place the order |

> **Note on synthetic data.** The dataset was produced by `scripts/generate_dataset.py`
> with a fixed random seed, so every number in this notebook is reproducible. Realistic
> behavioural patterns (festive peaks, discount-driven margin erosion, fashion returns,
> delivery-rating relationship) were built into the generator; the analysis below
> *discovers* them the same way it would on real data.

## 4. Technologies Used

| Library | Why it is used here |
|---|---|
| **NumPy** | Fast vectorised numeric work, percentiles, conditional flags |
| **Pandas** | Loading, cleaning, grouping, pivoting, time-series resampling |
| **Matplotlib** | Static charts for the report and presentation |
| **Seaborn** | Statistical charts (distributions, box plots, heatmaps) |
| **Plotly Express** | Interactive charts and the India geographic map |
| **Jupyter Notebook** | The analysis environment itself |

In [2]:
# step-1 Data collection

In [3]:
import numpy as np                 
import pandas as pd    
import seaborn as sns 
import matplotlib.pyplot as plt
import plotly.express as px  
import plotly.graph_objects as go
import json, os, warnings
from pathlib import Path

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")


In [4]:
df_raw = pd.read_csv('india_ecommerce_sales.csv')

In [5]:
df_raw.head()

,Order_ID,Order_Date,Customer_ID,Customer_Name,Gender,Age,Age_Group,City,State,Region,Latitude,Longitude,Product_ID,Product_Name,Category,Sub_Category,Quantity,Unit_Price,Discount_Percentage,Sales,Cost,Profit,Profit_Margin,Payment_Method,Order_Status,Delivery_Days,Customer_Rating,Customer_Segment,Device_Type
0,ORD101152,2024-03-21,CUST13416,Tanvi Rao,Female,23.00,18-25,Kochi,Kerala,South,9.93,76.27,P1042,Tata Tea Premium 1kg,Grocery,Beverages,2,645.40,7.70,"1,191.71","1,069.15",122.56,10.28,UPI,Delivered,5.00,4.20,Premium,Mobile App
1,ORD107594,2025-01-31,CUST12988,Swapnil Chatterjee,Female,44.00,36-45,Gurugram,Haryana,North,28.46,77.03,P1067,Blue Star 1T Window AC,Appliances,Air Conditioners,2,"36,248.75",15.60,"61,199.04","60,540.81",658.23,1.08,Credit Card,Delivered,3.00,4.50,Corporate,Mobile App
2,ORD109386,2025-05-15,CUST11826,Karan Deshmukh,Female,38.00,36-45,NaN,Gujarat,West,21.17,72.83,P1058,Data Science Handbook,Books,Academic,3,"1,733.50",7.90,"4,791.23","3,946.73",844.50,17.63,UPI,Delivered,2.00,4.80,Regular,Mobile Web
3,ORD102186,2024-05-21,CUST12825,Swapnil Bhat,Male,32.00,26-35,Mumbai,Maharashtra,West,19.08,72.88,P1041,Fortune Sunflower Oil 5L,Grocery,Staples,3,"1,373.75",4.00,"3,955.20","3,640.56",314.64,7.96,Wallet,Delivered,3.00,5.00,Premium,Tablet
4,ORD113618,2025-11-08,CUST12560,Imran Sharma,Female,42.00,36-45,Kanpur,Uttar Pradesh,North,26.45,80.33,P1015,W Printed Palazzo Set,Fashion,Women's Clothing,2,"2,260.70",25.40,"3,373.77","2,501.81",-495.79,-14.70,Cash on Delivery,Returned,5.00,2.90,Regular,Mobile App


In [6]:
df_raw.shape

(15120, 29)

In [7]:
# step-2 Data cleaning

In [10]:
for i in df_raw.columns:
    print(i)

Order_ID
Order_Date
Customer_ID
Customer_Name
Gender
Age
Age_Group
City
State
Region
Latitude
Longitude
Product_ID
Product_Name
Category
Sub_Category
Quantity
Unit_Price
Discount_Percentage
Sales
Cost
Profit
Profit_Margin
Payment_Method
Order_Status
Delivery_Days
Customer_Rating
Customer_Segment
Device_Type


In [12]:
df_raw.isnull().sum()

Order_ID                 0
Order_Date               0
Customer_ID              0
Customer_Name            0
Gender                 151
Age                    231
Age_Group                0
City                    91
State                    0
Region                   0
Latitude                 0
Longitude                0
Product_ID               0
Product_Name             0
Category                 0
Sub_Category             0
Quantity                 0
Unit_Price               0
Discount_Percentage      0
Sales                    0
Cost                     0
Profit                   0
Profit_Margin            0
Payment_Method         121
Order_Status             0
Delivery_Days          303
Customer_Rating        625
Customer_Segment         0
Device_Type              0
dtype: int64

In [13]:
df_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 15120 entries, 0 to 15119
Data columns (total 29 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Order_ID             15120 non-null  str    
 1   Order_Date           15120 non-null  str    
 2   Customer_ID          15120 non-null  str    
 3   Customer_Name        15120 non-null  str    
 4   Gender               14969 non-null  str    
 5   Age                  14889 non-null  float64
 6   Age_Group            15120 non-null  str    
 7   City                 15029 non-null  str    
 8   State                15120 non-null  str    
 9   Region               15120 non-null  str    
 10  Latitude             15120 non-null  float64
 11  Longitude            15120 non-null  float64
 12  Product_ID           15120 non-null  str    
 13  Product_Name         15120 non-null  str    
 14  Category             15120 non-null  str    
 15  Sub_Category         15120 non-null  str    
 1

**What `.info()` tells us**

* The file has **15,120 rows**, but several columns have fewer non-null values -
  `Customer_Rating`, `Delivery_Days`, `Age`, `Gender`, `Payment_Method` and `City`
  all contain missing data.
* `Order_Date` was read as **object (text)**, not as a date. Until we convert it we cannot
  do any time-series work.
* `Age` and `Delivery_Days` are floats only because missing values force pandas to use
  `float64`; conceptually they are whole numbers.

In [14]:
df_raw.describe().T

,count,mean,std,min,25%,50%,75%,max
Age,"14,889.00",34.29,10.59,1.00,27.00,34.00,41.00,200.00
Latitude,"15,120.00",20.36,6.07,8.52,13.08,20.30,26.45,30.90
Longitude,"15,120.00",77.55,4.10,72.57,73.86,77.10,78.49,91.74
Quantity,"15,120.00",1.95,4.59,-3.00,1.00,1.00,2.00,206.00
Unit_Price,"15,120.00","7,268.84","15,442.20",122.40,776.89,"1,910.14","5,191.51","890,296.75"
Discount_Percentage,"15,120.00",15.25,8.48,0.00,9.10,15.10,21.10,49.20
Sales,"15,120.00","8,168.42","15,406.42",109.87,"1,264.30","2,453.35","6,590.12","204,868.73"
Cost,"15,120.00","7,472.79","15,171.63",105.82,994.78,"1,958.74","5,279.10","219,886.83"
Profit,"15,120.00",542.32,"1,785.56","-23,504.19",53.63,337.51,873.48,"23,402.87"
Profit_Margin,"15,120.00",14.33,15.96,-51.78,3.73,14.78,26.14,51.39


**Red flags already visible in `.describe()`**

| Column | Suspicious value | Why it matters |
|---|---|---|
| `Quantity` | minimum is **negative** and maximum is in the hundreds | Negative units are impossible; huge quantities are data-entry glitches |
| `Age` | minimum near **1**, maximum **200** | Not a valid customer age |
| `Customer_Rating` | maximum above **5** | The rating scale is 1-5 |
| `Delivery_Days` | maximum near **60** | Almost certainly stuck/lost shipments |
| `Unit_Price` | maximum far above the highest catalogue price | Price-feed error |

We will handle each of these explicitly in the cleaning section.

In [15]:
# how many unique values each has and which value appears most often.
df_raw.describe(include="object").T

C:\Users\Admin\AppData\Local\Temp\ipykernel_16460\536132865.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  df_raw.describe(include="object").T


,count,unique,top,freq
Order_ID,15120,15000,ORD100264,2
Order_Date,15120,731,2025-10-05,52
Customer_ID,15120,3086,CUST12879,37
Customer_Name,15120,1212,Shruti Iyer,65
Gender,14969,6,Male,8060
Age_Group,15120,5,26-35,5019
City,15029,25,Mumbai,1454
State,15120,20,Maharashtra,3024
Region,15120,6,South,4835
Product_ID,15120,67,P1043,391


In [16]:
print("Unique values per column:")
df_raw.nunique().sort_values(ascending=False)

Unique values per column:


Order_ID               15000
Sales                  14837
Cost                   14807
Profit                 14477
Unit_Price             13953
Profit_Margin           5757
Customer_ID             3086
Customer_Name           1212
Order_Date               731
Discount_Percentage      420
Product_Name              67
Product_ID                67
Age                       54
Customer_Rating           39
Sub_Category              33
Quantity                  28
Longitude                 25
City                      25
Latitude                  25
State                     20
Delivery_Days             19
Category                  15
Payment_Method            12
Gender                     6
Region                     6
Age_Group                  5
Order_Status               4
Customer_Segment           4
Device_Type                4
dtype: int64

In [17]:
# handle null values

In [18]:
df_raw.dropna(inplace=True,ignore_index=True)

In [19]:
df_raw.isnull().sum().sum()

np.int64(0)

In [21]:
df_raw.shape

(13649, 29)

In [22]:
# change data types of date column 

In [23]:
df_raw['Order_Date']=pd.to_datetime(df_raw['Order_Date'])

In [25]:
df_raw.dtypes

Order_ID                          str
Order_Date             datetime64[us]
Customer_ID                       str
Customer_Name                     str
Gender                            str
Age                           float64
Age_Group                         str
City                              str
State                             str
Region                            str
Latitude                      float64
Longitude                     float64
Product_ID                        str
Product_Name                      str
Category                          str
Sub_Category                      str
Quantity                        int64
Unit_Price                    float64
Discount_Percentage           float64
Sales                         float64
Cost                          float64
Profit                        float64
Profit_Margin                 float64
Payment_Method                    str
Order_Status                      str
Delivery_Days                 float64
Customer_Rat

In [27]:
df_raw['Age']=df_raw['Age'].astype('int64')
df_raw['Delivery_Days']=df_raw['Delivery_Days'].astype('int64')

In [29]:
df_raw.dtypes

Order_ID                          str
Order_Date             datetime64[us]
Customer_ID                       str
Customer_Name                     str
Gender                            str
Age                             int64
Age_Group                         str
City                              str
State                             str
Region                            str
Latitude                      float64
Longitude                     float64
Product_ID                        str
Product_Name                      str
Category                          str
Sub_Category                      str
Quantity                        int64
Unit_Price                    float64
Discount_Percentage           float64
Sales                         float64
Cost                          float64
Profit                        float64
Profit_Margin                 float64
Payment_Method                    str
Order_Status                      str
Delivery_Days                   int64
Customer_Rat

In [ ]:
def get_unique(df):
    for i 